# Pharmagen — 2. Building the drug library

Everything the drug tower reads is produced offline by **Modephere**, a separate
package this project depends on. Nothing here touches the network: drugs are
resolved against a **staged ChEMBL release** (`data/chembl/chembl_37`, 22,631
molecules) that travels in the data bundle.

This notebook builds a small library live, then shows the two artifacts
Pharmagen adds on top: **ADMET-AI profiles** and the **HGVS catalog**.

A build produces, per compound:

1. a resolved identity (ChEMBL parent ID ↔ InChIKey, with aliases),
2. a standardized structure (largest fragment, rule-based protonation),
3. conformers ranked by MMFF94s energy,
4. a molecular graph and a pharmacophore graph, stored as `safetensors`.

In [ ]:
# --- Environment -----------------------------------------------------------
# Colab runs Python 3.13; Pharmagen and Modephere need 3.14. colab_setup lets
# uv provision 3.14 + the locked dependencies and starts a kernel there. Every
# cell below marked %%pgen runs in that 3.14 kernel and keeps its state.
REPO_URL = "https://github.com/Aderfi/Pgen_dev.git"
BRANCH = "modephere-imp"

# The data bundle (scripts/pack_colab_bundle.py, ~190 MB): a Google Drive path,
# or an https URL to a release asset. Ignored when running locally.
BUNDLE = "/content/drive/MyDrive/pharmagen_colab_bundle.zip"
# BUNDLE = "https://github.com/Aderfi/Pgen_dev/releases/download/demo/pharmagen_colab_bundle.zip"

import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO = Path("/content/dev_Pgen")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
else:
    REPO, BUNDLE = Path.cwd().parent, None  # notebooks/ -> repository root

sys.path.insert(0, str(REPO / "notebooks"))
import colab_setup

kernel = colab_setup.prepare(REPO, BUNDLE)

## A live build

Five drugs, into a throwaway directory so the real library is untouched. One of
them (`CHEMBL1141`) is chosen deliberately: it is an inorganic compound, and the
build is expected to **reject** it. A library build that silently dropped such
inputs would be worse than one that records why.

In [ ]:
%%pgen
import logging
import tempfile
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from modephere.library import LibraryBuilder

from src.config import get_settings
from src.data.library.bridge import chembl_index, library_build_config, library_store

root = Path(tempfile.mkdtemp(prefix="pgen-minilib-"))

# Point Modephere at the throwaway directory; everything else (the staged
# ChEMBL release, the PGx exports) stays as configured. bridge.py is the only
# place in the project that configures Modephere.
settings = get_settings().modephere.model_copy(
    update={"library": root / "library", "cache": root / "cache", "logs": root / "logs"}
)

CHEMBL_IDS = [
    "CHEMBL1771",     # clopidogrel  - the textbook CYP2C19 case
    "CHEMBL25",       # aspirin
    "CHEMBL1200829",  # gluconolactone
    "CHEMBL1141",     # an inorganic compound - expected to fail
    "CHEMBL112",      # acetaminophen
]

config = library_build_config(chembl_ids=CHEMBL_IDS, settings=settings)
summary = LibraryBuilder(config).run()
print("\n", summary)

Four built, one rejected. What the build wrote:

In [ ]:
%%pgen
import polars as pl

pl.Config.set_tbl_width_chars(220)
pl.Config.set_fmt_str_lengths(44)

store = library_store(settings)

print("Files written:", sorted(p.name for p in (root / "library").iterdir()), "\n")

print(store.read_table("compounds").select(
    "chembl_id", "title", "num_heavy_atoms", "formal_charge", "working_smiles"
))

print("\nChEMBL ID -> InChIKey (how the project keys every drug):")
for chembl, inchikey in sorted(chembl_index(store).items()):
    print(f"  {chembl:14} {inchikey}")

print("\nRejections, with the reason recorded:")
print(pl.read_parquet(root / "library" / "failures.parquet").select(
    "key", "stage", "category", "reason"
))

### Conformers

Pharmacophore features depend on 3D geometry, so the build embeds and minimizes
conformers and keeps them ranked by energy. Inference averages its prediction
over the three lowest-energy conformers.

In [ ]:
%%pgen
conformers = pl.read_parquet(root / "library" / "conformers.parquet")
compounds = store.read_table("compounds").select("inchikey", "title")

print(
    conformers.join(compounds, on="inchikey")
    .group_by("title")
    .agg(
        n=pl.len(),
        min_energy=pl.col("energy").min().round(2),
        max_rel=pl.col("relative_energy").max().round(2),
        converged=pl.col("converged").all(),
    )
    .sort("title")
)

print("\nClopidogrel's three lowest-energy conformers:")
print(
    conformers.join(compounds, on="inchikey")
    .filter(pl.col("title") == "CLOPIDOGREL")
    .sort("energy")
    .head(3)
    .select("conformer_id", "stereoisomer", "energy", "relative_energy", "force_field")
)

---

## The full production library

The five-compound run above uses the exact same pipeline that built the full
production store in `data/library/`. That build resolved PharmGKB's drug
vocabulary against the staged ChEMBL release, standardized structures at pH 7.4,
embedded conformer ensembles with ETKDGv3 + MMFF94s, and recorded every
compound that could not be built in `failures.parquet`.

In [ ]:
%%pgen
import json
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from src.config import get_settings

lib_dir = get_settings().paths.library
manifest = json.loads((lib_dir / "build_manifest.json").read_text())
full_compounds = pl.read_parquet(lib_dir / "compounds.parquet")
full_conformers = pl.read_parquet(lib_dir / "conformers.parquet")
full_failures = pl.read_parquet(lib_dir / "failures.parquet")
atc_members = pl.read_parquet(lib_dir / "pgx" / "atc_members.parquet")

print("Build versions :", manifest["versions"])
print(f"Compounds built: {full_compounds.height:,} (with 3D conformers: {full_conformers['inchikey'].n_unique():,})")
print(f"Total conformers: {full_conformers.height:,} (median {int(full_conformers.group_by('inchikey').len()['len'].median())} per compound)")
print(f"ATC drug classes: {atc_members['atc_code'].n_unique():,} codes covering {atc_members['chembl_id'].n_unique():,} ChEMBL drugs\n")

print("Recorded rejections / fallbacks across the full build:")
failure_summary = (
    full_failures.group_by("stage", "category")
    .agg(n=pl.len(), example_reason=pl.col("reason").first())
    .sort("n", descending=True)
)
print(failure_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Heavy-atom distribution
heavy = full_compounds["num_heavy_atoms"].to_numpy()
axes[0].hist(heavy, bins=35, color="#2b5c8f", edgecolor="white")
axes[0].axvline(np.median(heavy), color="#d95f02", linestyle="--", label=f"median = {int(np.median(heavy))}")
axes[0].set_title(f"Heavy atoms per compound (n = {len(heavy):,})")
axes[0].set_xlabel("heavy atoms")
axes[0].set_ylabel("compounds")
axes[0].legend()

# 2. Conformers per compound
conf_counts = full_conformers.group_by("inchikey").len()["len"].to_numpy()
axes[1].hist(conf_counts, bins=25, color="#388e3c", edgecolor="white")
axes[1].set_title(f"3D conformers per compound (total = {full_conformers.height:,})")
axes[1].set_xlabel("conformers stored")
axes[1].set_ylabel("compounds")

# 3. Relative energy window of stored conformers
rel_e = full_conformers["relative_energy"].drop_nulls().to_numpy()
axes[2].hist(rel_e, bins=35, color="#7570b3", edgecolor="white")
axes[2].set_title("MMFF94s relative energy (kcal/mol)")
axes[2].set_xlabel("ΔE above lowest-energy conformer")
axes[2].set_ylabel("conformers")

plt.tight_layout()
plt.show()

---

## ADMET profiles

Pharmagen adds one artifact of its own per compound: an **ADMET-AI** profile —
41 predicted properties covering absorption, distribution, metabolism,
excretion and toxicity, including the CYP enzymes that dominate
pharmacogenetics. These become a per-molecule feature vector in the drug tower.

On CPU this takes a few seconds for a handful of molecules.

In [ ]:
%%pgen
import time

from src.data.library.admet import compute_admet_table

rows = store.read_table("compounds").select("inchikey", "working_smiles")

start = time.perf_counter()
admet = compute_admet_table(rows.iter_rows())
print(f"\nADMET-AI: {admet.height} compounds in {time.perf_counter() - start:.1f} s on CPU")

named = admet.join(compounds, on="inchikey").select(
    "title", "CYP2C19_Veith", "CYP2D6_Veith", "CYP3A4_Veith", "hERG", "DILI", "BBB_Martins"
)
print(named.sort("title"))

Read the clopidogrel row: a high `CYP2C19_Veith` score, which is exactly the
interaction that makes clopidogrel the canonical pharmacogenetic example — it is
a prodrug that CYP2C19 must activate, so poor metabolizers get less of the active
drug. The model is given this signal on the drug side, and the patient's
`CYP2C19` variants on the genotype side.

The profile has 41 columns; the full width is what the tower consumes:

In [ ]:
%%pgen
from src.data.library.admet import DRUG_ADMET_DIM

print(f"DRUG_ADMET_DIM = {DRUG_ADMET_DIM}")
print(f"admet_profile.parquet columns = {admet.width} (inchikey + {admet.width - 1} properties)\n")

properties = [c for c in admet.columns if c != "inchikey"]
for i in range(0, len(properties), 4):
    print("  " + "".join(f"{p:<34}" for p in properties[i : i + 4]))

---

## Pharmacophore descriptor benchmark

Modephere perceives six non-overlapping pharmacophore families at pH 7.4 —
`Donor`, `Acceptor`, `PosIonizable`, `NegIonizable`, `Aromatic` and `Hydrophobe` —
and encodes pair counts across topological distance bins into the 258-dim
`pharm_global` vector alongside the 3D point graph.

`reports/pharmacophores/` holds the background-similarity benchmark over 370
drugs (3,000 random pairs) discussed in
`reports/Pharmacophore descriptors for drug separation.md`:

- Counting 1-point feature families alone (`family_tanimoto`) gives unrelated
  drugs a high background similarity (median 0.42, 95th percentile 0.71),
  because most drug-like molecules share donors, acceptors and aromatic rings.
- Separating pairs across topological distance bins (`pair_tanimoto`) cuts the
  random background nearly threefold (median 0.14), bringing pharmacophore
  selectivity closer to circular fingerprints (`morgan_tanimoto`, ECFP4) while
  retaining scaffold-hopping reach.

In [ ]:
%%pgen
pharm_files = sorted((get_settings().paths.reports / "pharmacophores").glob("compare_baseline_*.json"))
bench = json.loads(pharm_files[-1].read_text())

print(f"Benchmark file: {pharm_files[-1].name} "
      f"({bench['n_molecules']} molecules, {bench['n_pairs']:,} random pairs)\n")

header = f"{'descriptor / metric':22} {'median':>8} {'mean':>8} {'p95':>8} {'p99':>8}"
print(header)
print("-" * len(header))
for name, values in bench["values"].items():
    arr = np.asarray(values)
    print(f"{name:22} {np.median(arr):8.3f} {arr.mean():8.3f} "
          f"{np.percentile(arr, 95):8.3f} {np.percentile(arr, 99):8.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
palette = [
    ("family_tanimoto", "1-point family Tanimoto", "#d95f02"),
    ("global_cosine", "Modephere pharm_global cosine (258d)", "#7570b3"),
    ("pair_tanimoto", "2-point pair-distance Tanimoto", "#1b9e77"),
    ("morgan_tanimoto", "ECFP4 (Morgan2) Tanimoto", "#2b5c8f"),
    ("gobbi_tanimoto", "Gobbi Pharm2D (39,972 bits)", "#666666"),
]
for key, label, color in palette:
    ax.hist(bench["values"][key], bins=40, density=True, alpha=0.45, label=label, color=color)
ax.set_title("Random-pair background similarity distributions (3,000 drug pairs)")
ax.set_xlabel("pairwise similarity")
ax.set_ylabel("density")
ax.legend()
plt.tight_layout()
plt.show()

---

## The HGVS catalog

The genotype tower is keyed by HGVS, but pharmacogenetic sources are not: they
speak in rsIDs, star alleles (`CYP2C19*2`) and PharmGKB annotation IDs. The
catalog is the bridge. It is assembled from Modephere's PGx exports plus
`data/dicts/star_alleles.tsv`, and rebuilt here from scratch.

In [ ]:
%%pgen
import time

from src.data.genotype.catalog import build_hgvs_catalog

settings_all = get_settings()

start = time.perf_counter()
catalog = build_hgvs_catalog(
    pgx_export=settings_all.modephere.pgx_export,
    star_alleles_tsv=settings_all.paths.data / "dicts" / "star_alleles.tsv",
)
elapsed = time.perf_counter() - start

print(f"\nBuilt in {elapsed:.1f} s")
print(f"  {len(catalog.variants):>6} HGVS variants")
print(f"  {len(catalog.genes()):>6} genes")
print(f"  {len(catalog.labels):>6} labels (rsIDs and star alleles) -> HGVS")
print(f"  {len(catalog.annotations):>6} PharmGKB annotations -> HGVS")

What a resolution looks like, for the two best-known `CYP2C19` alleles:

In [ ]:
%%pgen
for label in ("CYP2C19*2", "CYP2C19*17", "CYP2D6*4"):
    hgvs = catalog.labels.get(label)
    print(f"{label:12} -> {len(hgvs)} variant(s)")
    for h in hgvs:
        info = catalog.info(h)
        terms = ", ".join(info.so_terms) if info.so_terms else "-"
        print(f"{'':14} {h}   [{terms}]")
    print()

print("An rsID resolves the same way:")
print("  rs1045642 ->", catalog.labels.get("rs1045642"))

from src.data.genotype.haplotype_function import FUNC_STATUSES

components = [*FUNC_STATUSES, "activity_score", "pgx_known"]
print("\nStar-allele function vectors (from data/dicts/star_alleles.tsv):")
print(f"\n  {'':12} " + " ".join(f"{c[:11]:>12}" for c in components))
for allele in ("*1", "*2", "*17", "*4"):
    vector = catalog.functions.vector_for("CYP2C19", allele)
    print(f"  CYP2C19{allele:5} " + " ".join(f"{v:>12.1f}" for v in vector))

The function vector hands the model a small amount of curated expert knowledge
directly, instead of making it infer the effect from variant positions: `*2` is
no-function with activity 0.0, `*17` is increased-function with activity 1.0.

The last component matters. `*1` — the reference allele — is **not** in the TSV,
so it comes back as all zeros with `pgx_known = 0`. That mask is what lets the
model distinguish *"this haplotype has no function"* from *"nobody has curated a
call for this haplotype"*. Without it, both would look like a zero vector, and
the absence of evidence would be indistinguishable from evidence of absence.
`*4` is in the TSV, so it resolves normally.

---

## Summary

A library build, start to finish, in seconds:

| Step | Produced |
|---|---|
| Resolve | ChEMBL parent ID ↔ InChIKey, with aliases |
| Standardize | largest fragment, rule-based protonation — rejects non-molecules |
| Conformers | energy-ranked 3D structures (MMFF94s) |
| Featurize | molecular graph + pharmacophore graph, as `safetensors` |
| ADMET | 41 predicted properties per compound |
| Catalog | rsID / star allele / annotation → HGVS, 16,271 variants |

The full library in the bundle holds 2,715 compounds, built the same way.
Notebook `03` opens these artifacts and shows the graphs the model receives.